# EasyTest 手动测试

用于交互式验证 HTTP、RPC、数据库和 UI 适配器，复用 `tests/fixtures/project/` 中的框架测试数据。默认严格离线，不访问外部服务。日常接口演示与 Mock 示例见 `examples/jsonplaceholder/`。

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from easytest.notebook import NotebookSession

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
ROOT = REPO_ROOT / "tests" / "fixtures" / "project"

session = NotebookSession(ROOT, run_mode="read", profile="offline-strict")
session.operations()

## 编译并查看 XLSX 用例

In [ ]:
case_file = "cases/scenario/account_flow.xlsx"
session.compile(case_file)
session.cases(case_file)

## 运行完整场景

In [ ]:
results = session.run_case(case_file, case_id="scenario.account_flow")
results

## 单步手动请求

当前会话使用框架测试数据和严格离线 Profile。测试真实业务时，先将 `ROOT` 指向已配置接口的业务项目，再新建 `NotebookSession(ROOT, profile="live")` 并传 `mock=False`；数据库造数另需显式设置 `allow_db_write=True`。

In [ ]:
http_result = session.run_step(
    executor="http",
    operation="http.get_profile",
    request={"params": {"id": "user-001"}},
    mock="http_success",
)
http_result

In [ ]:
rpc_result = session.run_step(
    executor="rpc",
    operation="rpc.calculate_limit",
    request={"user_id": "user-001", "amount": "500.00"},
    mock="rpc_success",
)
rpc_result

In [ ]:
database_result = session.run_step(
    executor="database",
    operation="database.list_accounts",
    request={"parameters": {"user_id": "user-001"}},
    mock="database_rows",
)
database_result

## UI 请求与截图

In [ ]:
ui_result = session.run_step(
    executor="ui",
    operation="ui.open_dashboard",
    request={"user_id": "user-001"},
    mock="ui_dashboard",
)
ui_result

In [ ]:
image = ui_result.image()
session.close()
image